In [91]:
import numpy as np
import pandas as pd
from os import listdir
import re
from os.path import isfile, join

# Formatting the intensities

The aim of this section is to aggregate all the data into a unique csv file. I add the ROI as a column

In [278]:
path = '../IMC_Segmentation_results/all_data/'
directory = 'intensities/'
file_names = [f for f in listdir(path+directory) if isfile(join(path+directory, f))]
stripped_names = [re.sub(r'\.[^.]+$', '', name) for name in file_names]

For every intensity file, I associate the appropriate ROI

In [231]:
images = pd.read_csv(path+'images.csv',index_col=0)
images['filename'] = images.image
images['ROI'] = images.acquisition_description.str.extract(r'(ROI_\d+)')#extract ROI from string
images.image = images.image.str.replace( r'\.[^.]+$','',regex = True)#strip  file extension
images =images[~images.duplicated(subset = ['image','ROI'])]
common_set = set(stripped_names)&set(images.image)
images = images[images.image.isin(common_set)][['image','ROI']]
images.set_index('image',inplace=True)
images


,ROI
image,
Leap001_008,ROI_001
Leap001_009,ROI_002
Leap001_010,ROI_003
Leap002_003,ROI_007
Leap002_004,ROI_005
Leap002_005,ROI_004
Leap002_006,ROI_003
Leap002_007,ROI_002


I also add a column called ``source_file`` just to check in the future for batch effects. AnnoSpat doesn't care as long it is after the ROI.

In [282]:
merged_intensities = pd.DataFrame()
for file in common_set:
    df = pd.read_csv(path+directory+file+'.csv')
    df = df[df.columns[df.columns.str.find('-')>0]]#select only columns related to antibody and filter out empty channels
    df.columns = df.columns.str.replace(r'^.*?-','',regex = True)#remove the metals

    df['ROI'] = images.loc[file].values[0]
    df['source_file'] = file
    merged_intensities = pd.concat([merged_intensities,df],ignore_index=True)
print('shape of the dataframe:',merged_intensities.shape)
merged_intensities.head()

shape of the dataframe: (58031, 39)


,CD38,EGFR,p53,CD14,Tbet,CD16,CD163,Pan-keratin,CD11b,PD-L1,...,CD27,PD-L2,CD45RO,Alpha-SMA,Vimentin,CD31,DNA1,DNA2,ROI,source_file
0,0.525571,0.490078,0.764117,0.868680,0.639988,0.177013,0.062500,1.165243,1.428627,0.176200,...,0.144994,0.467670,1.169612,1.407416,0.506011,0.125198,4.204468,9.026807,ROI_007,Leap002_003
1,0.515276,0.324182,0.076923,0.305964,0.519610,0.085043,0.000000,1.743893,0.845461,0.370850,...,0.308680,0.413358,0.976917,1.002858,0.631692,0.313133,12.053872,18.935056,ROI_007,Leap002_003
2,0.436287,0.675673,0.561728,1.066378,0.475544,0.408479,0.186198,1.345208,1.613524,0.280647,...,0.590248,0.505237,0.686423,0.676052,0.496595,0.438376,9.899534,17.316983,ROI_007,Leap002_003
3,0.471603,1.317043,0.358976,1.175608,0.543862,0.375068,0.177496,3.742146,1.586432,0.531933,...,0.549276,0.451247,0.784658,0.819292,0.462176,0.589874,7.838971,12.750305,ROI_007,Leap002_003
4,0.694710,0.684521,0.289157,1.725331,0.676068,0.697254,0.130107,1.264105,2.430705,0.228462,...,0.504643,0.218492,1.419250,0.382099,0.308627,0.245315,4.029468,5.821431,ROI_007,Leap002_003


save the file

In [263]:
merged_intensities.to_csv('./processed_files/all_data_intensities.csv')

Concatenate together the region files which contains the spatial information for every cell

In [275]:
merged_regions = pd.DataFrame()
for file in common_set:
    df = pd.read_csv(path+'/regionprops/'+file+'.csv')
    df['source_file'] = file

    merged_regions = pd.concat([merged_regions,df.iloc[:,1:]],ignore_index=True)#remove the object column
    
merged_regions.to_csv('./processed_files/all_data_regions.csv')

# Creating the signature
I format the signature file to work with AnnoSpat

In [7]:
#signature = pd.read_csv('../IMC_SegmentationResults/IMC_SegmentationResults/IMCCelltypeResults/cell_type_matrix.csv').set_index('Marker')
signature = pd.read_csv('../IMC_Segmentation_results/IMCCelltypeResults/cell_type_matrix.csv').set_index('Marker')
signature.head()


,Cancer,CD163p_Mac [M2],CD163n_Mac [M1],Naive CD4 T cells,Memory T cells,Regulatory T cells,Naive CD8 T cells,Memory CD8 T cells,Naive B cells,Memory B cells,...,CI_Monocyte,Int_Monocyte,nonCI_Monocyte,Endothelial,Fibroblasts,B7H4+ cancer cells,Proliferative cells?,p53+ cells?,Neutrophil&monocyte,Other cells
Marker,,,,,,,,,,,,,,,,,,,,,
CD38,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EGFR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
p53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
CD14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,1.0,-1.0,-1.0,NaN,NaN,NaN,NaN,-1.0,NaN
Tbet,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [182]:
new = {}
for column in signature.columns:
    protein_2_add = signature.index[~signature[column].isna()]#series of protein to use as markers
    a =signature[column][protein_2_add]
    new[column] = list(protein_2_add.where(a>0,protein_2_add.astype(str)+'-'))
signature =pd.DataFrame.from_dict(new,orient = 'index').T

In [183]:
signature.to_csv('processed_files/signature.csv')    

In [189]:
signature

,Cancer,CD163p_Mac [M2],CD163n_Mac [M1],Naive CD4 T cells,Memory T cells,Regulatory T cells,Naive CD8 T cells,Memory CD8 T cells,Naive B cells,Memory B cells,...,CI_Monocyte,Int_Monocyte,nonCI_Monocyte,Endothelial,Fibroblasts,B7H4+ cancer cells,Proliferative cells?,p53+ cells?,Neutrophil&monocyte,Other cells
0,Pan-keratin,CD163,CD163-,CD38,CD38,CD38,CD38,CD38,CD38,CD38,...,CD14,CD14,CD14-,CD14-,Pan-keratin-,Pan-keratin,Ki-67,p53,CD14-,Pan-keratin-
1,E-Cadherin,Pan-keratin-,Pan-keratin-,CD11b-,CD11b-,CD11b-,CD11b-,CD11b-,CD11b-,CD11b-,...,CD16-,CD16,CD16,CD16-,CD45-,E-Cadherin,None,Ki-67,CD16,CD45-
2,Beta-Catenin,CD68,CD68,CD45,CD45,CD45,CD45,CD45,CD45,CD45,...,Pan-keratin-,Pan-keratin-,Pan-keratin-,Pan-keratin-,Alpha-SMA,Beta-Catenin,None,None,CD11b,Alpha-SMA-
3,None,None,None,CD44-,CD44,CD44,CD44-,CD44,FOXP3-,FOXP3-,...,CD68-,CD68-,CD68-,CD68-,Vimentin,B7-H4,None,None,CD45,CD31-
4,None,None,None,FOXP3-,FOXP3-,FOXP3,FOXP3-,FOXP3-,CD20,CD20,...,CD20-,CD20-,CD20-,CD20-,CD31-,None,None,None,None,DNA1
5,None,None,None,CD4,CD4,CD4,CD4-,CD4-,CD3-,CD3-,...,CD3-,CD3-,CD3-,CD3-,None,None,None,None,None,None
6,None,None,None,CD20-,CD20-,CD20-,CD20-,CD20-,CD27-,CD27,...,None,None,None,CD31,None,None,None,None,None,None
7,None,None,None,CD8a-,CD8a-,CD8a-,CD8a,CD8a,None,None,...,None,None,None,None,None,None,None,None,None,None
8,None,None,None,CD3,CD3,CD3,CD3,CD3,None,None,...,None,None,None,None,None,None,None,None,None,None
9,None,None,None,CD27,CD27,CD27,CD27-,CD27,None,None,...,None,None,None,None,None,None,None,None,None,None


COde below run AnnoSpat to perform cell type annotation

In [267]:
!python AnnoSpat_main/AnnoSpat/run.py -i processed_files/all_data_intensities.csv  -m processed_files/signature.csv  -o ./output -f CD38 -l DNA2 -r ROI -a '[99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5, 99.5,99.5]'


=====Function chosen: ANNOTATION=====

Starting cell type annotation.....

-----Estimating cell types using Semi supervised clustering-----

-------------------------Done-------------------------

Time taken: 0.02073277235031128seconds

Estimating cell types...
-----Estimating cell types using ELM-----

-------------------------Done-------------------------

Time taken: 0.0663349191347758seconds

Saving Annotations in: /home/giuseppe/Downloads/Annospat_attempt/output/trte_labels_numericLabels_ELM_IMC_T1D_AnnoSpat.csv
-------------------------Done-------------------------

